In [ ]:
# AI Assistant: Plain-Language Weather Guidance from METAR Data

This notebook builds on the query workflow from `0_Intro` and `2_Interactive_Viz`.
It takes structured METAR observations and translates them into **plain-language,
actionable weather guidance for non-expert users** — such as farmers and ranchers.

A key design principle: when data is missing or sparse, the assistant communicates
that limitation **honestly** rather than fabricating a confident answer. This
reflects responsible-AI practice for non-expert audiences.

In [5]:
import duckdb
import pandas as pd
from datetime import datetime, timedelta

# --- Pick a station and a time ---
station = "DEN"       
year, month, day, hour = 2026, 1, 1, 0 

time_1 = datetime(year, month, day, hour)
time_0 = time_1 - timedelta(hours=1)

# Build the data URLs (handles the year-boundary edge case)
YYYY_0, YYYY_1 = time_0.strftime("%Y"), time_1.strftime("%Y")
base = "https://data.source.coop/dynamical/asos-parquet"
if YYYY_0 == YYYY_1:
    URLs = [f"{base}/year={YYYY_1}/data.parquet"]
else:
    URLs = [f"{base}/year={YYYY_0}/data.parquet",
            f"{base}/year={YYYY_1}/data.parquet"]

# Query just our one station for that hour
df = duckdb.execute("""
    SELECT valid, station, name, country, tmpf, dwpf, sknt, p01i, longitude, latitude
    FROM read_parquet($1, hive_partitioning=true)
    WHERE station = $2
      AND valid BETWEEN $3 AND $4
    ORDER BY valid
""", [URLs, station, time_0, time_1]).fetchdf()

df

,valid,station,name,country,tmpf,dwpf,sknt,p01i,longitude,latitude
0,2025-12-31 23:53:00+00:00,DEN,DENVER INTNL ARPT,US,57.0,15.0,4.0,0.0,-104.6575,39.8328


In [6]:
def extract_observation(df):
    """Pull the key weather fields from the query result.
    Returns a dict; missing values become None so we can handle them honestly."""
    if df is None or len(df) == 0:
        return None  # no data at all

    row = df.iloc[-1]  # most recent observation in the window

    def safe(val):
        return None if pd.isna(val) else val

    return {
        "station": safe(row.get("station")),
        "name": safe(row.get("name")),
        "country": safe(row.get("country")),
        "time": str(safe(row.get("valid"))),
        "temp_f": safe(row.get("tmpf")),
        "dewpoint_f": safe(row.get("dwpf")),
        "wind_kt": safe(row.get("sknt")),
        "precip_in": safe(row.get("p01i")),
    }

obs = extract_observation(df)
obs

{'station': 'DEN',
 'name': 'DENVER INTNL ARPT',
 'country': 'US',
 'time': '2025-12-31 23:53:00+00:00',
 'temp_f': np.float64(57.0),
 'dewpoint_f': np.float64(15.0),
 'wind_kt': np.float64(4.0),
 'precip_in': np.float64(0.0)}

In [7]:
def generate_guidance_rule_based(obs):
    """Plain-language weather guidance for non-expert (e.g. agricultural) users.
    Honestly flags missing data instead of guessing."""
    if obs is None:
        return ("No weather observations were found for this station and time. "
                "I can't provide guidance without data — please try a nearby "
                "station or a different time.")

    where = obs["name"] or obs["station"] or "this location"
    lines = [f"Weather guidance for {where} (observed {obs['time']}):", ""]

    # Temperature
    if obs["temp_f"] is not None:
        t = obs["temp_f"]
        note = ""
        if t <= 32:
            note = " Freezing conditions — protect sensitive crops and livestock water."
        elif t >= 90:
            note = " Hot — ensure livestock have shade and water; avoid midday field labor."
        lines.append(f"• Temperature: {t:.0f}°F.{note}")
    else:
        lines.append("• Temperature: not reported at this station for this hour.")

    # Wind
    if obs["wind_kt"] is not None:
        mph = obs["wind_kt"] * 1.15
        note = " Too windy for safe spraying." if mph >= 10 else " Calm enough for spraying."
        lines.append(f"• Wind: about {mph:.0f} mph.{note}")
    else:
        lines.append("• Wind: not reported — do not assume calm conditions before spraying.")

    # Precipitation
    if obs["precip_in"] is not None:
        p = obs["precip_in"]
        note = " Recent rainfall — fields may be wet." if p > 0 else " No recent precipitation recorded."
        lines.append(f"• Precipitation (last hour): {p:.2f} in.{note}")
    else:
        lines.append("• Precipitation: not reported for this hour.")

    lines.append("")
    lines.append("Note: This guidance is based only on the available observations above. "
                 "Where a value is missing, no assumption has been made.")
    return "\n".join(lines)

print(generate_guidance_rule_based(obs))

Weather guidance for DENVER INTNL ARPT (observed 2025-12-31 23:53:00+00:00):

• Temperature: 57°F.
• Wind: about 5 mph. Calm enough for spraying.
• Precipitation (last hour): 0.00 in. No recent precipitation recorded.

Note: This guidance is based only on the available observations above. Where a value is missing, no assumption has been made.


In [ ]:
### Demonstration: honest handling of missing and partial data

A core feature of this assistant is that it never fabricates values. Below,
the first example shows the response when **no data** is available; the second
shows **partial data** (temperature present, wind missing), where the assistant
explicitly flags what it cannot confirm.

In [8]:
# Demonstrate honest handling of missing data:
# query a station/time unlikely to have data, and show the assistant does NOT fabricate.

obs_missing = None  # simulate "no observation found"
print(generate_guidance_rule_based(obs_missing))
print("\n" + "="*50 + "\n")

# Also show partial data: temperature present, wind missing
obs_partial = {
    "station": "TEST", "name": "Example Ranch", "country": "US",
    "time": "2026-01-01 00:00", "temp_f": 28.0,
    "dewpoint_f": None, "wind_kt": None, "precip_in": None
}
print(generate_guidance_rule_based(obs_partial))

No weather observations were found for this station and time. I can't provide guidance without data — please try a nearby station or a different time.


Weather guidance for Example Ranch (observed 2026-01-01 00:00):

• Temperature: 28°F. Freezing conditions — protect sensitive crops and livestock water.
• Wind: not reported — do not assume calm conditions before spraying.
• Precipitation: not reported for this hour.

Note: This guidance is based only on the available observations above. Where a value is missing, no assumption has been made.
